## Import and load ##

In [4]:
# cell 1: imports & basic config
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# paths
root = Path('.')  # notebook is in TREES/
images_dir = root / "images"

Device: cpu


### Compute classes and split dataset

In [5]:
basic_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
full_ds = datasets.ImageFolder(str(images_dir), transform=basic_tf)
class_to_idx = full_ds.class_to_idx
idx_to_class = {v:k for k,v in class_to_idx.items()}
num_classes = len(class_to_idx)
print("Found classes:", num_classes, idx_to_class)

Found classes: 5 {0: 'koivu', 1: 'kuusi', 2: 'lehmus', 3: 'pihlaja', 4: 'vaahtera'}


In [6]:
labels = [s[1] for s in full_ds.samples]
train_idx, val_idx = train_test_split(
    np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=42)

### Train and test

In [ ]:
# Training and test transforms
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

from torch.utils.data import Subset

train_ds = Subset(datasets.ImageFolder(str(images_dir), transform=train_tf), train_idx)
test_ds  = Subset(datasets.ImageFolder(str(images_dir), transform=test_tf),  val_idx)  # rename val -> test

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("Train/test sizes:", len(train_ds), len(test_ds))


In [ ]:
model = models.resnet50(pretrained=True)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)  # adjust for your number of species
model = model.to(device)


In [ ]:
for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            preds = out.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    return correct / total


In [ ]:
n_epochs = 8

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    test_acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}/{n_epochs} - Loss: {avg_loss:.4f} - Test Accuracy: {test_acc:.4f}")


In [ ]:
# unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
n_epochs_ft = 1

for epoch in range(n_epochs_ft):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    test_acc = evaluate(model, test_loader, device)
    print(f"FT Epoch {epoch+1}/{n_epochs_ft} - Loss: {avg_loss:.4f} - Test Accuracy: {test_acc:.4f}")

torch.save(model.state_dict(), "resnet50/resnet50_finetuned.pth")


In [ ]:
def predict_image(img_path, model, idx_to_class, device):
    model.eval()
    img = Image.open(img_path).convert("RGB")
    x = test_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x)
        probs = torch.nn.functional.softmax(out, dim=1)
        p, pred = torch.max(probs, dim=1)
    return idx_to_class[int(pred.item())], float(p.item())

# example
print(predict_image("images/pihlaja/pihlaja9.jpg", model, idx_to_class, device))
